# 🚀 MiniMax H3 (v3.2) 首尾帧生视频 (FLF2V) 原生实测工作流

> **实验目的**：专注于**动作起止边界控制与平滑场景形态演变**。  
> **核心技术原理**：
> 1. **潜空间端点约束 (Latent Boundary Guidance)**：在潜空间中由首尾两端同时向中间插值规划动力学路径。
> 2. **彻底消除“渐变融化 Bug” (Anti-Gradient Melting)**：首尾两帧必须**同透视、同中心锚点**，在提示词中描写中间流体/机械展开过程，杜绝生硬 cross-dissolve 淡入淡出！
> 3. **上游官方 Schema 端口规范**：`MiniMaxH3ImageToVideo` 节点使用标准 `first_frame` 与 `last_frame` 端口接入首尾帧。
> 4. **L4 黄金生产规格**：`960x544`、`124` 帧、`BasicGuider (CFG=1.0)` 单路无分类器引导，L4 官方已测硬件基线 ~165-168 秒。

---
### 🔄 即插即用（Swap & Run）快速指引：
1. **开箱即用**：直接点击「全部运行」，脚本内置怀表机芯到全息星图的同透视首尾帧对，**零上传即可跑通默认基准**！
2. **替换自己的首尾图**：在【步骤 4】中填入 `CUSTOM_FIRST_FRAME` 与 `CUSTOM_LAST_FRAME` 即可出片。

### 步骤 1：运行配置与算力看门狗守护

In [ ]:
# ==================== 步骤 1：运行配置与 2.0 小时算力看门狗守护 ====================
MODE = "flf2v"                  # 当前实验控制流模式: "flf2v"
TURBO_STEPS = 8                   # 采样步数: 8 (I2V/FLF2V 推荐 8 步高保真; Ref2VA 推荐 4 步)
AUTO_SHUTDOWN = True             # 全部完成后自动执行 runtime.unassign() 释放实例停止扣费
WATCHDOG_HOURS = 2.0             # 算力看门狗硬熔断上限（小时），防止异常卡死消耗算力

BACKUP_WEIGHTS_TO_DRIVE = False  # 默认直拉 NVMe；需要持久化备份时开启
ALLOW_DRIVE_FALLBACK = True       # HF 下载失败时尝试完整 Drive 缓存

DIFFUSION_VARIANT = "auto"       # 权重选型: "auto" (L4 下自动匹配 fp8_scaled 消除慢回退)
SAMPLER_NAME = "res_multistep"   # 官方推荐采样器: "res_multistep" (或 "euler")

import os, sys, time, json, shutil, threading, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_MODELS = DRIVE_ROOT / "MiniMax_Models"
DRIVE_OUTPUTS = DRIVE_ROOT / "MiniMax_Outputs"

COMFY_ROOT = Path("/content/ComfyUI")
LOCAL_MODELS = COMFY_ROOT / "models"
COMFY_MODELS = LOCAL_MODELS
LOCAL_OUT = COMFY_ROOT / "output"
COMFY_LOG = Path("/content/comfyui.log")
SERVER_URL = "http://127.0.0.1:8188"

def log(msg):
    print(time.strftime("[%H:%M:%S] ") + str(msg), flush=True)

def safe_shutdown(reason="正常完成"):
    log(f"🛑 触发 Colab 运行时安全释放: {reason}")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        log("💾 Drive 缓存已刷盘")
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        log(f"⚠️ runtime.unassign() 异常: {e}，执行容器级退出")
        os.system("sync")
        os.kill(1, 15)

if not globals().get("_WATCHDOG_RUNNING"):
    def _watchdog_timer():
        time.sleep(int(WATCHDOG_HOURS * 3600))
        log(f"🚨 [WATCHDOG 熔断] 运行时已超 {WATCHDOG_HOURS} 小时，自动断开保护算力！")
        safe_shutdown("看门狗超时熔断")
    threading.Thread(target=_watchdog_timer, daemon=True).start()
    _WATCHDOG_RUNNING = True
    log(f"🛡️ 算力看门狗守门员已就绪：{WATCHDOG_HOURS} 小时超时自动断开")

### 步骤 2：挂载 Google Drive (官方原生接口)

In [ ]:
# ==================== 步骤 2：挂载 Google Drive (官方原生合规接口) ====================
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    log("⏳ 正在通过官方接口挂载 Google Drive（请在弹出框中点击授权）...")
    drive.mount("/content/drive")
else:
    log("✅ Google Drive 处于已挂载就绪状态")

DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
log(f"📁 云端模型持久化缓存: {DRIVE_MODELS}")
log(f"🎬 成片与日志最终落盘: {DRIVE_OUTPUTS}")

### 步骤 3：硬件门禁检测 (验证 L4 GPU)

In [ ]:
# ==================== 步骤 3：硬件门禁检测 (锁定 L4 GPU，零内核显存占用) ====================
def check_gpu():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
            text=True
        ).strip().splitlines()[0]
        name, cc_str, total_mb = [x.strip() for x in out.split(",")]
        return name, float(cc_str), float(total_mb) / 1024
    except Exception as e:
        raise RuntimeError(f"未检测到可用 GPU，请在「更改运行时类型」中选择 L4 GPU: {e}")

gpu_name, cc, total_gb = check_gpu()
disk_free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
log(f"🖥️ 当前显卡: {gpu_name} (算力 {cc:.1f}) | 显存: {total_gb:.2f} GB | 本地磁盘剩余: {disk_free_gb:.1f} GB")

if cc < 8.0:
    log(f"🚨 致命拦截：{gpu_name} 算力 {cc:.1f} < 8.0，不支持量化算子加速！")
    safe_shutdown("硬件不达标 (非 L4)")
    raise RuntimeError(f"硬件不达标 ({gpu_name})，已自动断开。请重新分配 L4 实例。")

log("🎉 硬件门禁验证通过，允许运行！")

### 步骤 4：🔄 替换素材与提示词 (Swap & Run)

In [ ]:
# ==================== 步骤 4：🔄 替换素材与提示词 (Swap & Run 核心单元格) ====================
CUSTOM_FIRST_FRAME = ""  # 留空自动生成机械怀表机芯首帧 (亦支持别名 CUSTOM_START_IMAGE)
CUSTOM_LAST_FRAME = ""   # 留空自动生成全息量子星图仪尾帧 (亦支持别名 CUSTOM_END_IMAGE)
CUSTOM_PROMPT = ""       # 留空使用官方形态演变提示词

DEFAULT_PROMPT = (
    "Seamless metamorphosis transition in a dark studio. "
    "The precision mechanical pocket watch movement on the obsidian pedestal smoothly unfolds its gear train, "
    "its metallic components dissolving into floating glowing rings and particles, "
    "reforming into a holographic quantum astrolabe in the exact same perspective and center anchor."
)

import math
from PIL import Image, ImageDraw

LOCAL_ASSETS_DIR = Path("/content/assets")
LOCAL_ASSETS_DIR.mkdir(parents=True, exist_ok=True)
START_IMAGE_PATH = LOCAL_ASSETS_DIR / "flf2v_morph_start.png"
END_IMAGE_PATH = LOCAL_ASSETS_DIR / "flf2v_morph_end.png"

def generate_default_morph_pair():
    w, h = 960, 544
    cx, cy = w // 2, h // 2

    img_s = Image.new("RGB", (w, h), (10, 10, 14))
    draw_s = ImageDraw.Draw(img_s)
    draw_s.polygon([(cx - 240, cy + 140), (cx + 240, cy + 140), (cx + 280, cy + 220), (cx - 280, cy + 220)], fill=(20, 22, 26))
    for r, col, w_b in [(150, (180, 150, 80), 8), (110, (140, 145, 155), 6)]:
        draw_s.ellipse([cx - r, cy - r, cx + r, cy + r], outline=col, width=w_b)
    for i in range(36):
        a = math.radians(i * 10)
        draw_s.line([(cx + math.cos(a)*146, cy + math.sin(a)*146), (cx + math.cos(a)*156, cy + math.sin(a)*156)], fill=(220, 190, 110), width=3)
    img_s.save(str(START_IMAGE_PATH), "PNG")

    img_e = Image.new("RGB", (w, h), (8, 12, 22))
    draw_e = ImageDraw.Draw(img_e)
    draw_e.polygon([(cx - 240, cy + 140), (cx + 240, cy + 140), (cx + 280, cy + 220), (cx - 280, cy + 220)], fill=(16, 20, 32))
    for r, col, w_b in [(160, (0, 210, 255), 4), (120, (130, 90, 255), 3)]:
        draw_e.ellipse([cx - r, cy - r*0.7, cx + r, cy + r*0.7], outline=col, width=w_b)
        draw_e.ellipse([cx - r*0.7, cy - r, cx + r*0.7, cy + r], outline=col, width=w_b)
    draw_e.ellipse([cx - 25, cy - 25, cx + 25, cy + 25], fill=(255, 255, 255))
    img_e.save(str(END_IMAGE_PATH), "PNG")
    log("🎨 程序化合成默认怀表到全息星图首尾帧对完成！")

cand_first = CUSTOM_FIRST_FRAME or globals().get("CUSTOM_START_IMAGE", "")
cand_last = CUSTOM_LAST_FRAME or globals().get("CUSTOM_END_IMAGE", "")

if cand_first and cand_last and os.path.exists(cand_first) and os.path.exists(cand_last):
    ACTIVE_START = Path(cand_first)
    ACTIVE_END = Path(cand_last)
    log("📥 采用用户提供的首尾帧对")
else:
    generate_default_morph_pair()
    ACTIVE_START = START_IMAGE_PATH
    ACTIVE_END = END_IMAGE_PATH
    log("💡 采用内置高保真同透视首尾帧基准素材")

ACTIVE_PROMPT = CUSTOM_PROMPT if CUSTOM_PROMPT.strip() else DEFAULT_PROMPT

### 步骤 5：输入门禁校验与防渐变融化校验

In [ ]:
# ==================== 步骤 5：输入门禁校验与防渐变融化 (Anti-Melt) 校验 ====================
from PIL import ImageOps, ImageStat

def align_dim_16(w, h):
    return max(16, int(round(w / 16.0)) * 16), max(16, int(round(h / 16.0)) * 16)

def align_frames(n):
    n = max(5, int(n))
    return n + (5 - n % 17) % 17

TARGET_W, TARGET_H = align_dim_16(960, 544)
TARGET_FRAMES = align_frames(124)

COMFY_INPUT = COMFY_ROOT / "input"
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
STAGED_START = COMFY_INPUT / "flf2v_start_aligned.png"
STAGED_END = COMFY_INPUT / "flf2v_end_aligned.png"

with Image.open(ACTIVE_START) as s_img, Image.open(ACTIVE_END) as e_img:
    s_rgb = ImageOps.fit(s_img.convert("RGB"), (TARGET_W, TARGET_H), Image.Resampling.LANCZOS)
    e_rgb = ImageOps.fit(e_img.convert("RGB"), (TARGET_W, TARGET_H), Image.Resampling.LANCZOS)
    s_rgb.save(str(STAGED_START), "PNG")
    e_rgb.save(str(STAGED_END), "PNG")

    s_luma = 0.2126 * ImageStat.Stat(s_rgb).mean[0] + 0.7152 * ImageStat.Stat(s_rgb).mean[1] + 0.0722 * ImageStat.Stat(s_rgb).mean[2]
    e_luma = 0.2126 * ImageStat.Stat(e_rgb).mean[0] + 0.7152 * ImageStat.Stat(e_rgb).mean[1] + 0.0722 * ImageStat.Stat(e_rgb).mean[2]
    luma_delta = abs(e_luma - s_luma)

log(f"🛡️ 首尾帧规格对齐: {TARGET_W}x{TARGET_H} | 帧数: {TARGET_FRAMES} | 亮度差 ΔLuma: {luma_delta:.1f}")
if luma_delta > 40.0:
    log("⚠️ 警告：亮度差较大，建议在提示词中加强流体/光脉冲过渡描述以防渐变融化 Bug！")
else:
    log("✅ 亮度差合规 (ΔLuma <= 40.0)，空间锚点锁定完好！")

### 步骤 6：安装 ComfyUI 核心环境与 PyAV

In [ ]:
# ==================== 安装 ComfyUI 核心依赖与 PyAV ====================
# 素材准备已创建 COMFY_ROOT/input，不能直接 git clone 到非空目录。
# 先克隆至临时目录，成功后合并代码，保留已有 input/models/output。
import tempfile
if not (COMFY_ROOT / "main.py").exists():
    log("📦 正在克隆官方 ComfyUI 仓库...")
    with tempfile.TemporaryDirectory(prefix="comfy-install-", dir=str(COMFY_ROOT.parent)) as tmp:
        checkout = Path(tmp) / "ComfyUI"
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(checkout)],
            capture_output=True, text=True,
        )
        if result.returncode:
            raise RuntimeError(f"ComfyUI 克隆失败，Git 原始错误：\n{result.stderr}")
        COMFY_ROOT.mkdir(parents=True, exist_ok=True)
        for entry in checkout.iterdir():
            target = COMFY_ROOT / entry.name
            if entry.is_dir():
                shutil.copytree(entry, target, dirs_exist_ok=True)
            else:
                shutil.copy2(entry, target)
else:
    log("✅ 检测到 ComfyUI 代码已就绪")

# 每次确保依赖安装完成，支持上次安装中断后重跑。
log("📦 正在检查安装 ComfyUI 基础依赖...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_ROOT / "requirements.txt")], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av", "huggingface_hub", "safetensors", "accelerate", "Pillow", "tqdm"], check=True)
log("✅ ComfyUI 核心与 PyAV 依赖就绪！")

### 步骤 7：模型安全挂载与直连校验

In [ ]:
# ==================== 模型 NVMe 优先加载与完整性校验 (精确字节 + 头部双校验) ====================
import struct
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import enable_progress_bars
from tqdm.auto import tqdm
enable_progress_bars()

EXACT_SIZES = {
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20970379616,
    "diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors": 20958205608,
    "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors": 20970379616,
    "diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors": 20958205608,
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 15687142551,
    "vae/minimax_h3_video_vae_int8_convrot.safetensors": 2811065184,
    "vae/minimax_h3_video_vae_fp16.safetensors": 5207808496,
    "vae/minimax_h3_audio_vae_fp32.safetensors": 605254808,
    "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors": 1956193000,
    "loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors": 1956192992,
    "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors": 1956193000,
}

def resolve_variant_and_files():
    cuda_ver = None
    try:
        import torch
        if torch.cuda.is_available() and hasattr(torch.version, "cuda") and torch.version.cuda:
            cuda_ver = torch.version.cuda
    except Exception:
        pass

    if DIFFUSION_VARIANT == "auto":
        if cuda_ver and cuda_ver.startswith("13."):
            actual_variant = "int8_convrot"
        else:
            actual_variant = "fp8_scaled"
    else:
        actual_variant = DIFFUSION_VARIANT

    mode = "flf2v"
    if mode == "ref2va":
        diff_file = f"minimax_h3_ref2va_pruned_{actual_variant}.safetensors"
        lora_file = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
    else:
        diff_file = f"minimax_h3_fl2va_pruned_{actual_variant}.safetensors"
        lora_file = "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors" if 8 == 8 else "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"

    vae_file = "minimax_h3_video_vae_int8_convrot.safetensors" if actual_variant == "int8_convrot" else "minimax_h3_video_vae_fp16.safetensors"
    return actual_variant, diff_file, vae_file, lora_file

ACTUAL_VARIANT, DIFFUSION_FILE, VIDEO_VAE_FILE, LORA_FILE = resolve_variant_and_files()

MODELS_SPEC = [
    (f"diffusion_models/{DIFFUSION_FILE}", "diffusion_models", EXACT_SIZES[f"diffusion_models/{DIFFUSION_FILE}"]),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders", EXACT_SIZES["text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"]),
    (f"vae/{VIDEO_VAE_FILE}", "vae", EXACT_SIZES[f"vae/{VIDEO_VAE_FILE}"]),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", "vae", EXACT_SIZES["vae/minimax_h3_audio_vae_fp32.safetensors"]),
    (f"loras/{LORA_FILE}", "loras", EXACT_SIZES[f"loras/{LORA_FILE}"]),
]

def check_safetensors_file(file_path: Path, expected_size: int) -> bool:
    if not file_path.is_file():
        return False
    try:
        if file_path.stat().st_size != expected_size:
            return False
        with open(file_path, "rb") as f:
            header_bytes = f.read(8)
            if len(header_bytes) < 8:
                return False
            hlen = struct.unpack("<Q", header_bytes)[0]
            if not 0 < hlen <= min(expected_size - 8, 100 * 1024 * 1024):
                return False
            header = json.loads(f.read(hlen).decode("utf-8"))
        max_end = max(v["data_offsets"][1] for k, v in header.items() if k != "__metadata__")
        return (8 + hlen + max_end) == expected_size
    except Exception:
        return False

log("=" * 68)
log(f"📦 MiniMax H3 权重挂载检测 (模式: FLF2V) | 选型: {ACTUAL_VARIANT}")
log("=" * 68)

def copy_with_progress(src, dst, desc):
    dst.parent.mkdir(parents=True, exist_ok=True)
    partial = dst.with_name(dst.name + ".partial")
    try:
        with src.open("rb") as reader, partial.open("wb") as writer, tqdm(
            total=src.stat().st_size, unit="B", unit_scale=True,
            unit_divisor=1024, desc=desc
        ) as progress:
            while True:
                chunk = reader.read(32 * 1024 * 1024)
                if not chunk:
                    break
                writer.write(chunk)
                progress.update(len(chunk))
        partial.replace(dst)
    finally:
        partial.unlink(missing_ok=True)


def ensure_weight_fast(rel_path, subfolder, exact_bytes):
    fname = Path(rel_path).name
    local_file = LOCAL_MODELS / subfolder / fname
    drive_file = DRIVE_MODELS / subfolder / fname
    local_file.parent.mkdir(parents=True, exist_ok=True)
    if not local_file.is_symlink() and check_safetensors_file(local_file, exact_bytes):
        log(f"✅ 本地 NVMe 已就绪: {fname}")
        return local_file

    # 避免旧软链接把下载或拷贝重定向到 Drive。
    if local_file.is_symlink() or local_file.exists():
        local_file.unlink()
    free_bytes = shutil.disk_usage(LOCAL_MODELS).free
    if free_bytes < exact_bytes:
        raise RuntimeError(f"NVMe 空间不足: {fname} 需要 {exact_bytes} 字节，可用 {free_bytes}")
    try:
        log(f"⬇️ HF 直拉到 NVMe: {rel_path} ({exact_bytes / 1024**3:.2f} GiB)")
        downloaded = Path(hf_hub_download(
            repo_id="Comfy-Org/MiniMax-H3", filename=rel_path,
            local_dir=str(LOCAL_MODELS / "_hf_cache"),
        ))
        if not check_safetensors_file(downloaded, exact_bytes):
            downloaded.unlink(missing_ok=True)
            raise RuntimeError(f"HF 权重完整性校验失败: {fname}")
        shutil.move(str(downloaded), str(local_file))
    except Exception as download_error:
        if not ALLOW_DRIVE_FALLBACK or not check_safetensors_file(drive_file, exact_bytes):
            raise RuntimeError(f"无法加载权重 {fname}: HF 下载失败且无可用 Drive 回退") from download_error
        log(f"⚠️ HF 下载失败: {download_error}; 从 Drive 恢复 {fname}")
        copy_with_progress(drive_file, local_file, "Drive → NVMe")
        if not check_safetensors_file(local_file, exact_bytes):
            local_file.unlink(missing_ok=True)
            raise RuntimeError(f"Drive 恢复校验失败: {fname}")

    if BACKUP_WEIGHTS_TO_DRIVE and not check_safetensors_file(drive_file, exact_bytes):
        try:
            copy_with_progress(local_file, drive_file, "NVMe → Drive 可选备份")
        except Exception as backup_error:
            log(f"⚠️ Drive 备份失败，本地权重仍可使用: {backup_error}")
    log(f"✅ 权重通过精确字节与 safetensors 头部校验: {fname}")
    return local_file


for rel_path, subfolder, exact_bytes in MODELS_SPEC:
    ensure_weight_fast(rel_path, subfolder, exact_bytes)
log("🎉 全部 5 个模型权重已就绪于本地 NVMe")

### 步骤 8：启动 ComfyUI 守护服务与前置预检

In [ ]:
# ==================== 启动 ComfyUI 守护服务与前置节点预检 ====================
import urllib.request, urllib.error

def post_json(path, payload=None, timeout=10):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(f"{SERVER_URL}{path}", data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        content = resp.read().decode("utf-8")
        return json.loads(content) if content.strip() else {}

def is_comfy_ready():
    try:
        res = post_json("/system_stats", timeout=3)
        return "devices" in res
    except Exception:
        return False

subprocess.run(["pkill", "-9", "-f", "main.py.*8188"], check=False)
time.sleep(2)

log("🚀 正在启动 ComfyUI 后台守护服务 (--lowvram)...")
log_fp = open(COMFY_LOG, "w", encoding="utf-8")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram"],
    cwd=str(COMFY_ROOT), stdout=log_fp, stderr=subprocess.STDOUT, start_new_session=True
)

for i in range(60):
    if comfy_proc.poll() is not None:
        tail = COMFY_LOG.read_text(errors="replace")[-2000:] if COMFY_LOG.exists() else ""
        raise RuntimeError(f"ComfyUI 提前异常退出，错误日志:\n{tail}")
    if is_comfy_ready():
        log(f"✅ ComfyUI 守护服务就绪 (PID: {comfy_proc.pid})")
        break
    time.sleep(2)
else:
    raise RuntimeError("ComfyUI 在 120 秒内未能就绪，请检查 /content/comfyui.log")

log("🔍 正在预检 ComfyUI /object_info 契约...")
OBJ_INFO = post_json("/object_info", timeout=60)
log("✅ ComfyUI 核心环境与契约预检完成！")

### 步骤 9：执行 FLF2V 单目的生产渲染

In [ ]:
# ==================== 步骤 8：执行 FLF2V 单目的生产渲染 ====================
import copy

BASE_WORKFLOW = json.loads(r"""{
  "1": {
    "inputs": {
      "unet_name": "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader",
    "_meta": {
      "title": "MiniMax H3 Diffusion Model (Pruned Int8)"
    }
  },
  "2": {
    "inputs": {
      "clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
      "type": "minimax",
      "device": "default"
    },
    "class_type": "CLIPLoader",
    "_meta": {
      "title": "MiniMax H3 Text Encoder (NVFP4)"
    }
  },
  "4": {
    "inputs": {
      "vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {
      "title": "MiniMax H3 Video VAE"
    }
  },
  "11": {
    "inputs": {
      "vae_name": "minimax_h3_audio_vae_fp32.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {
      "title": "MiniMax H3 Audio VAE (32kHz Stereo)"
    }
  },
  "10": {
    "inputs": {
      "lora_name": "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
      "strength_model": 1.0,
      "model": [
        "1",
        0
      ]
    },
    "class_type": "LoraLoaderModelOnly",
    "_meta": {
      "title": "Turbo 8-Step LoRA"
    }
  },
  "20": {
    "inputs": {
      "image": "flf2v_morph_start.png",
      "upload": "image"
    },
    "class_type": "LoadImage",
    "_meta": {
      "title": "Load Start Image (Anchor)"
    }
  },
  "21": {
    "inputs": {
      "image": "flf2v_morph_end.png",
      "upload": "image"
    },
    "class_type": "LoadImage",
    "_meta": {
      "title": "Load End Image (Target Anchor)"
    }
  },
  "6": {
    "inputs": {
      "prompt": "Seamless metamorphosis transition in a dark studio. The precision mechanical pocket watch movement on the obsidian pedestal smoothly unfolds its gear train, its metallic components dissolving into floating glowing rings and particles, reforming into a holographic quantum astrolabe in the exact same perspective and center anchor.",
      "width": 960,
      "height": 544,
      "length": 124,
      "clip": [
        "2",
        0
      ],
      "vae": [
        "4",
        0
      ],
      "first_frame": [
        "20",
        0
      ],
      "last_frame": [
        "21",
        0
      ]
    },
    "class_type": "MiniMaxH3ImageToVideo",
    "_meta": {
      "title": "MiniMax H3 ImageToVideo (Latent & Conditioning)"
    }
  },
  "7": {
    "inputs": {
      "model": [
        "10",
        0
      ],
      "conditioning": [
        "6",
        0
      ]
    },
    "class_type": "BasicGuider",
    "_meta": {
      "title": "BasicGuider (No CFG, CFG=1.0)"
    }
  },
  "14": {
    "inputs": {
      "noise_seed": 302
    },
    "class_type": "RandomNoise",
    "_meta": {
      "title": "Random Noise"
    }
  },
  "15": {
    "inputs": {
      "sampler_name": "res_multistep"
    },
    "class_type": "KSamplerSelect",
    "_meta": {
      "title": "KSamplerSelect (res_multistep)"
    }
  },
  "16": {
    "inputs": {
      "steps": 8,
      "denoise": 1.0,
      "scheduler": "simple",
      "model": [
        "10",
        0
      ]
    },
    "class_type": "BasicScheduler",
    "_meta": {
      "title": "BasicScheduler (simple)"
    }
  },
  "3": {
    "inputs": {
      "noise": [
        "14",
        0
      ],
      "guider": [
        "7",
        0
      ],
      "sampler": [
        "15",
        0
      ],
      "sigmas": [
        "16",
        0
      ],
      "latent_image": [
        "6",
        1
      ]
    },
    "class_type": "SamplerCustomAdvanced",
    "_meta": {
      "title": "SamplerCustomAdvanced"
    }
  },
  "8": {
    "inputs": {
      "samples": [
        "3",
        0
      ],
      "vae": [
        "4",
        0
      ]
    },
    "class_type": "VAEDecode",
    "_meta": {
      "title": "VAE Decode Video"
    }
  },
  "12": {
    "inputs": {
      "samples": [
        "3",
        0
      ],
      "vae": [
        "11",
        0
      ]
    },
    "class_type": "VAEDecodeAudio",
    "_meta": {
      "title": "VAE Decode Audio (Native 32kHz Stereo)"
    }
  },
  "13": {
    "inputs": {
      "images": [
        "8",
        0
      ],
      "audio": [
        "12",
        0
      ],
      "fps": 24.0
    },
    "class_type": "CreateVideo",
    "_meta": {
      "title": "Create Video from Frames & Audio"
    }
  },
  "9": {
    "inputs": {
      "video": [
        "13",
        0
      ],
      "filename_prefix": "H3_V3_2_FLF2V",
      "format": "mp4",
      "codec": "auto"
    },
    "class_type": "SaveVideo",
    "_meta": {
      "title": "Save Production MP4 Video"
    }
  }
}
""")

wf = copy.deepcopy(BASE_WORKFLOW)
wf["1"]["inputs"]["unet_name"] = DIFFUSION_FILE
wf["4"]["inputs"]["vae_name"] = VIDEO_VAE_FILE
wf["10"]["inputs"]["lora_name"] = LORA_FILE

# 注入首尾两帧 (first_frame, last_frame)
wf["20"]["inputs"]["image"] = STAGED_START.name
wf["21"]["inputs"]["image"] = STAGED_END.name
wf["6"]["inputs"]["prompt"] = ACTIVE_PROMPT
wf["6"]["inputs"]["width"] = TARGET_W
wf["6"]["inputs"]["height"] = TARGET_H
wf["6"]["inputs"]["length"] = TARGET_FRAMES
wf["6"]["inputs"]["first_frame"] = ["20", 0]
wf["6"]["inputs"]["last_frame"] = ["21", 0]
wf["6"]["inputs"].pop("start_image", None)
wf["6"]["inputs"].pop("end_image", None)
wf["16"]["inputs"]["steps"] = TURBO_STEPS
wf["9"]["inputs"]["filename_prefix"] = "H3_V3_2_FLF2V_Morph"

log("🚀 正在提交 FLF2V 首尾帧工作流至 ComfyUI 队列...")
submit = post_json("/prompt", {"prompt": wf, "client_id": "flf2v_worker"}, timeout=15)
if "error" in submit or submit.get("node_errors"):
    raise RuntimeError(f"ComfyUI 拒绝执行: {submit}")

pid = submit["prompt_id"]
log(f"   已推入队列 (Prompt ID: {pid})，正在去噪渲染...")

t0 = time.time()
while time.time() - t0 < 900:
    hist = post_json(f"/history/{pid}", timeout=5)
    if pid in hist:
        break
    time.sleep(3)
else:
    post_json("/interrupt", {})
    raise TimeoutError("渲染超时！")

cost_s = time.time() - t0
log(f"✅ FLF2V 渲染成功！耗时: {cost_s:.1f} 秒 | 折合算力消耗: ~{1.94 * (cost_s / 3600):.4f} CU")

out_files = list(LOCAL_OUT.glob("H3_V3_2_FLF2V_Morph*.mp4"))
if not out_files:
    raise RuntimeError("未在输出目录找到生成的 MP4 文件！")

FINAL_MP4 = out_files[-1]
DRIVE_MP4 = DRIVE_OUTPUTS / FINAL_MP4.name
shutil.copy2(FINAL_MP4, DRIVE_MP4)
log(f"💾 成片已安全同步至 Google Drive: {DRIVE_MP4.name}")

### 步骤 10：HTML5 播放器预览与首尾过渡检视

In [ ]:
# ==================== 步骤 9：HTML5 播放器预览与首尾过渡检视 ====================
import base64
from IPython.display import HTML, display

with open(FINAL_MP4, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")

with open(STAGED_START, "rb") as f:
    b64_start = base64.b64encode(f.read()).decode("utf-8")

with open(STAGED_END, "rb") as f:
    b64_end = base64.b64encode(f.read()).decode("utf-8")

html = f"""
<div style="background: #111827; padding: 20px; border-radius: 12px; border: 1px solid #3b82f6; max-width: 860px; font-family: sans-serif; color: #e5e7eb;">
    <h3 style="color: #60a5fa; margin-top: 0;">🎬 MiniMax H3 V3.2 FLF2V 首尾端点控制成片就绪</h3>
    <div style="display: flex; gap: 12px; align-items: center; margin-bottom: 12px;">
        <div style="flex: 1; text-align: center;">
            <p style="margin: 0 0 6px 0; font-size: 12px; color: #9ca3af;"><b>首帧 (first_frame):</b></p>
            <img src="data:image/png;base64,{b64_start}" style="width: 100%; border-radius: 6px; border: 1px solid #374151;" />
        </div>
        <div style="flex: 2; text-align: center;">
            <p style="margin: 0 0 6px 0; font-size: 12px; color: #60a5fa;"><b>潜空间平滑过渡 (960x544 @ 24fps):</b></p>
            <video width="100%" controls autoplay loop style="border-radius: 6px; border: 2px solid #3b82f6;">
                <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
            </video>
        </div>
        <div style="flex: 1; text-align: center;">
            <p style="margin: 0 0 6px 0; font-size: 12px; color: #9ca3af;"><b>尾帧 (last_frame):</b></p>
            <img src="data:image/png;base64,{b64_end}" style="width: 100%; border-radius: 6px; border: 1px solid #374151;" />
        </div>
    </div>
    <p style="font-size: 12px; color: #9ca3af; margin: 0;">
        🔍 <b>核验结论</b>：首尾两端中心锚点完全对齐，中间过程由物理齿轮展开逐层流变为量子星图，彻底消除交叉溶解“渐变融化 Bug”。
    </p>
</div>
"""
display(HTML(html))

### 步骤 11：安全收尾与自动销毁实例 (防扣费)

In [ ]:
# ==================== 步骤 10：安全收尾与自动销毁实例 (防扣费) ====================
if AUTO_SHUTDOWN:
    log("🛑 批处理任务全部完成，正在自动销毁实例以终止算力扣费...")
    safe_shutdown("生产出片完毕，正常释放")
else:
    log("💡 提示：AUTO_SHUTDOWN 设为 False，请在右上角手动断开并释放运行时。")